In [ ]:
## This script is for model training using APL framework and ResNet-50 features

In [ ]:
# import packages
import os, cv2
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

## check GPU
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

In [ ]:
# load data (first part)
## load shadow-free images
path_wd = '../' ## set working directory
img = cv2.imread(path_wd + 'output/images/shadow_free.jpg')
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) ## transform color channel

In [ ]:
# choose parameters
THRESH = 20 ## diameter threshold
n_c = 25 ## number of clusters

In [ ]:
# load data (second part)
## load clustering results
labels = pd.read_csv(path_wd + 'output/KMeans/labels_' + str(n_c) + '.csv', index_col=0).to_numpy().flatten()
## load relevance
rele_PREMON = pd.read_csv(path_wd + 'output/relevance/relevance_PREMON_' + str(THRESH) + '_' + str(n_c) + '.csv', index_col=0).to_numpy().flatten()
rele_CECSCH = pd.read_csv(path_wd + 'output/relevance/relevance_CECSCH_' + str(THRESH) + '_' + str(n_c) + '.csv', index_col=0).to_numpy().flatten()
rele_MANBID = pd.read_csv(path_wd + 'output/relevance/relevance_MANBID_' + str(THRESH) + '_' + str(n_c) + '.csv', index_col=0).to_numpy().flatten()

In [ ]:
# create training set
time_start = datetime.now()
print('Start:', time_start)
## create training patches
n_img = len(labels)
## create image arrays
x_dat = np.zeros((n_img, 100, 100, 3), dtype=np.uint8)
y_dat = np.zeros((n_img, 3))
## loop over the patches
for i in range(n_img):
    rr = i // 96
    cc = i % 96
    lab_cluster = labels[i]
    x_dat[i] = img[(rr*100):(rr*100+100), (cc*100):(cc*100+100)]
    y_dat[i] = np.array([rele_PREMON[lab_cluster], rele_CECSCH[lab_cluster], rele_MANBID[lab_cluster]])
## label rescale (This step is optional)
## rescale of feature labels, such as the cluster with largest palm relevance has label as 1
for i in range(3):
    y_dat[:, i] = y_dat[:, i] * (1 / np.max(y_dat[:, i]))
print('Finished:', datetime.now()-time_start)



In [ ]:
# build the model
## 定义Dataset类
class TreePatchDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        """
        images: numpy array [N, 100, 100, 3]
        labels: numpy array [N, 3]
        """
        self.images = images
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]  # [100, 100, 3]
        label = self.labels[idx]  # [3]
        
        if self.transform:
            # 转换为PIL Image然后应用transform
            image = Image.fromarray(image.astype('uint8'))
            image = self.transform(image)
        
        label = torch.FloatTensor(label)
        return image, label

## PyTorch预处理（等价于Keras的preprocess_input）
transform = transforms.Compose([
    transforms.Resize(100),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                       std=[0.229, 0.224, 0.225])
])

## 定义模型类
class APLModel(nn.Module):
    def __init__(self, num_classes=3):
        super(APLModel, self).__init__()
        # 加载预训练ResNet-50
        resnet = models.resnet50(pretrained=True)
        # 移除最后的全连接层（只保留到avgpool）
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        # GlobalAveragePooling已经包含在ResNet的avgpool中
        self.fc = nn.Linear(2048, num_classes)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        x = self.features(x)        # [batch, 2048, 1, 1]
        x = torch.flatten(x, 1)     # [batch, 2048]
        x = self.fc(x)              # [batch, 3]
        x = self.sigmoid(x)         # [batch, 3]
        return x

## load the ResNet model
resnet_model_base = APLModel(num_classes=3).to(device)

## compile the model (PyTorch不需要compile，直接定义优化器和损失)
criterion = nn.MSELoss()
optimizer = optim.Adam(resnet_model_base.parameters(), lr=1e-4)

In [ ]:
# parameter selection
## train-validation split
np.random.seed(2020)
n_img = x_dat.shape[0]
loc_train = np.random.choice(n_img, int(n_img * 0.8), replace=False)
loc_val = np.setdiff1d(np.arange(n_img), loc_train)
## start model training
time_start = datetime.now()
print('Start training:', time_start)
## 这个cell是用来测试的，真正的训练在下一个cell
# 如果要测试，可以取消下面的注释：
# train_dataset_test = TreePatchDataset(x_dat[loc_train], y_dat[loc_train], transform=transform)
# train_loader_test = DataLoader(train_dataset_test, batch_size=64, shuffle=True)
# for epoch in range(1):  # 测试1个epoch
#     for images, labels in train_loader_test:
#         images = images.to(device)
#         labels = labels.to(device)
#         outputs = resnet_model_base(images)
#         loss = criterion(outputs, labels)
#         break  # 只测试一个batch
#     break
print('Time for model training:', datetime.now()-time_start)

In [ ]:
# retrain the model with selected epochs
## build the model
model = APLModel(num_classes=3).to(device)
## compile the model (设置优化器和损失函数)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

## 创建DataLoader
train_dataset = TreePatchDataset(x_dat, y_dat, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_dataset = TreePatchDataset(x_dat[loc_val], y_dat[loc_val], transform=transform)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

## start model training
time_start = datetime.now()
print('Start training:', time_start)

num_epochs = 50
for epoch in range(num_epochs):
    # 训练阶段
    model.train()
    train_loss = 0.0
    train_mae = 0.0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        # 前向传播
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # 反向传播
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_mae += torch.mean(torch.abs(outputs - labels)).item()
    
    # 验证阶段
    model.eval()
    val_loss = 0.0
    val_mae = 0.0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            val_mae += torch.mean(torch.abs(outputs - labels)).item()
    
    # 打印统计（每个epoch）
    train_loss_avg = train_loss / len(train_loader)
    train_mae_avg = train_mae / len(train_loader)
    val_loss_avg = val_loss / len(val_loader)
    val_mae_avg = val_mae / len(val_loader)
    
    print(f'Epoch [{epoch+1}/{num_epochs}], '
          f'Train Loss: {train_loss_avg:.4f}, Train MAE: {train_mae_avg:.4f}, '
          f'Val Loss: {val_loss_avg:.4f}, Val MAE: {val_mae_avg:.4f}')

## save the model
save_path = path_wd + 'output/models/APL_' + str(THRESH) + '_' + str(n_c) + '.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'epoch': num_epochs,
    'train_loss': train_loss_avg,
    'val_loss': val_loss_avg
}, save_path)
print('Time for model training:', datetime.now()-time_start)